# Multi-Domain Crisis Resolver

A LangChain cookbook that orchestrates Sarvam's `sarvam-105b` reasoning model across multiple chains to resolve crises faced by India's informal-sector workers -- auto drivers, construction laborers, street vendors -- whose health, financial, and legal problems rarely arrive one at a time.

A hospitalization is also lost income. Lost income can become a loan default. A loan default can become a legal dispute over a vehicle or a home. Answering each domain in isolation misses exactly the interactions that make these situations a crisis rather than three unrelated problems.

Pipeline overview:
1. **Triage** -- `sarvam-105b` reads the free-text case and decides which of health, financial, and legal domains are actually in play (structured output; no domain is assumed relevant by default)
2. **Domain chains** -- only the domains triage marked relevant run, concurrently via LangChain's `RunnableParallel`, each producing a structured expert assessment grounded in real Indian welfare schemes and worker-protection laws
3. **Cross-domain synthesis** -- a second call to `sarvam-105b`, with `reasoning_effort="high"`, reasons in plain text over every domain assessment together to catch the interactions between them; a third, lightweight call then fits that reasoned answer into a structured plan
4. Save the full trace (triage, domain assessments, reasoning, and final plan) to `outputs/`

This notebook demonstrates LLM orchestration patterns -- structured output, dynamic parallel chains, reasoning-then-structure synthesis -- and the schemes and helplines referenced are for illustration. It is not a substitute for professional legal, medical, or financial advice; verify current scheme details and helpline numbers against official sources before acting on them.


In [ ]:
%pip install -r requirements.txt


## Setup

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable, RunnableParallel
from langchain_sarvamai import ChatSarvam
from pydantic import BaseModel, Field

load_dotenv()

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running."
    )

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Triage and the three domain chains only need to fill in a fixed schema, so the
# default (unset) reasoning effort keeps them fast and cheap.
llm = ChatSarvam(model="sarvam-105b")

# Cross-domain synthesis has to reason about how a health, financial, and legal fact
# interact -- exactly the multi-step reasoning reasoning_effort="high" is for.
reasoning_llm = ChatSarvam(model="sarvam-105b", reasoning_effort="high")


## Known resources

Every domain chain and the synthesis step are instructed to cite schemes and helplines only from this list. Grounding the prompt with real names and numbers, rather than trusting the model to recall them, is what keeps the output from hallucinating a scheme or a phone number that does not exist.

The entries below are illustrative and may go stale -- verify against official sources (ayushmanbharat.gov.in, eshram.gov.in, nalsa.gov.in) before relying on them.


In [ ]:
KNOWN_RESOURCES = {
    "health": [
        "Ayushman Bharat PM-JAY: free hospitalization cover up to INR 5 lakh/family/year "
        "for eligible low-income families. Helpline: 14555.",
        "Employees' State Insurance (ESI): medical care and cash benefits for workers "
        "formally registered with an ESI-contributing employer.",
    ],
    "financial": [
        "PM Jan Dhan Yojana (PMJDY): zero-balance bank account, needed to receive any "
        "government benefit transfer.",
        "PM Jeevan Jyoti Bima Yojana (PMJJBY): INR 2 lakh life cover for about INR "
        "436/year, auto-debited from a linked bank account.",
        "PM Suraksha Bima Yojana (PMSBY): INR 2 lakh accidental death/disability cover "
        "for about INR 20/year.",
        "PM Street Vendor's AtmaNirbhar Nidhi (PM SVANidhi): working-capital loans up "
        "to INR 50,000 for street vendors.",
        "e-Shram portal registration: unlocks accident insurance and access to "
        "unorganised-worker welfare schemes.",
    ],
    "legal": [
        "e-Shram / Universal Account Number (UAN) registration under the Code on Social "
        "Security, 2020: the basis for most unorganised-worker benefits.",
        "Building and Other Construction Workers (BOCW) Welfare Cess Act, 1996: "
        "registered construction workers are entitled to medical aid, accident "
        "assistance, and pension from the state BOCW welfare board.",
        "Motor Vehicles Act, 1988: mandates third-party insurance; a valid commercial "
        "license and permit protect an auto driver from vehicle seizure after an "
        "accident.",
        "National Legal Services Authority (NALSA): free legal aid for workers below "
        "the income threshold. Toll-free helpline: 15100.",
        "Shram Suvidha labour helpline for wage disputes and workplace grievances: "
        "1800-11-4880.",
    ],
}


## Define the crisis case

`WorkerProfile` and `CrisisCase` are the input contract every chain in this notebook shares. `CASE_HUMAN_MESSAGE` is the one human-turn template reused by triage and every domain chain, so the case facts are formatted identically everywhere they are read.


In [ ]:
class WorkerProfile(BaseModel):
    """Basic context about the worker facing the crisis."""

    occupation: str = Field(
        description="e.g. 'auto-rickshaw driver', 'construction laborer', 'street vendor'"
    )
    location: str = Field(description="City and state, e.g. 'Bengaluru, Karnataka'")
    monthly_income_range_inr: str = Field(description="e.g. '9000-12000'")
    dependents: int = Field(description="Number of people financially dependent on this worker")


class CrisisCase(BaseModel):
    """A worker's crisis, as reported in free text."""

    profile: WorkerProfile
    narrative: str = Field(description="Free-text description of what happened")

    def as_prompt_vars(self) -> dict[str, str | int]:
        """Flatten the case into the variables every chain's prompt template expects."""
        return {
            "occupation": self.profile.occupation,
            "location": self.profile.location,
            "monthly_income_range_inr": self.profile.monthly_income_range_inr,
            "dependents": self.profile.dependents,
            "narrative": self.narrative,
        }


# Shared by every chain that needs the raw case facts (triage and the three domain chains).
CASE_HUMAN_MESSAGE = (
    "human",
    "Occupation: {occupation}\nLocation: {location}\n"
    "Monthly income (INR): {monthly_income_range_inr}\nDependents: {dependents}\n\n"
    "Narrative:\n{narrative}",
)


## Step 1: Triage -- decide which domains apply

A single LangChain call with `with_structured_output` classifies the case against a `DomainTriage` schema before any domain-specific work happens. This is the dynamic-routing step: it keeps a purely financial crisis from paying for a health or legal chain it does not need.


In [ ]:
class DomainTriage(BaseModel):
    """Which domains this crisis actually touches, decided before any domain chain runs."""

    health_relevant: bool = Field(
        description="True if the narrative involves an injury, illness, or medical need"
    )
    health_reason: str = Field(description="One sentence justifying the health_relevant decision")
    financial_relevant: bool = Field(
        description="True if the narrative involves lost income, debt, or a lack of savings"
    )
    financial_reason: str = Field(description="One sentence justifying the financial_relevant decision")
    legal_relevant: bool = Field(
        description="True if the narrative involves a legal right, dispute, or risk of losing an asset or document"
    )
    legal_reason: str = Field(description="One sentence justifying the legal_relevant decision")


TRIAGE_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You triage crisis reports for informal-sector workers in India. Decide which of "
            "health, financial, and legal domains are genuinely relevant to the case below. "
            "Mark a domain relevant only if the narrative gives a concrete reason to involve "
            "it, not just because it could theoretically apply.",
        ),
        CASE_HUMAN_MESSAGE,
    ]
)

triage_chain = TRIAGE_PROMPT | llm.with_structured_output(DomainTriage)


## Step 2: Domain expert chains

Each domain gets its own schema, its own expert persona, and its own slice of `KNOWN_RESOURCES` -- a focused chain answers more precisely than one giant prompt asked to be a doctor, an accountant, and a paralegal at once. `_domain_chain` builds all three the same way so the only thing that differs between them is data, not code.


In [ ]:
class HealthAssessment(BaseModel):
    urgency: Literal["routine", "urgent", "emergency"] = Field(
        description="How quickly the health issue needs attention"
    )
    immediate_steps: list[str] = Field(description="Concrete actions for the next 24-48 hours")
    relevant_schemes: list[str] = Field(
        description="Health schemes or coverage, drawn only from the resources given, that apply here"
    )
    follow_up: list[str] = Field(description="Actions for the following weeks, e.g. documentation or enrolment")


class FinancialAssessment(BaseModel):
    income_impact: str = Field(description="One sentence on how this crisis affects income or savings")
    immediate_steps: list[str] = Field(description="Concrete actions for the next 24-48 hours")
    relevant_schemes: list[str] = Field(
        description="Financial schemes or protections, drawn only from the resources given, that apply here"
    )
    follow_up: list[str] = Field(description="Actions for the following weeks, e.g. applications or enrolment")


class LegalAssessment(BaseModel):
    rights_at_stake: list[str] = Field(description="Specific legal rights or protections the worker can invoke")
    immediate_steps: list[str] = Field(description="Concrete actions for the next 24-48 hours")
    relevant_schemes: list[str] = Field(
        description="Legal protections, registrations, or aid programs, drawn only from the resources given, that apply here"
    )
    follow_up: list[str] = Field(description="Actions for the following weeks, e.g. filings or registrations")


def _resource_block(domain: str) -> str:
    """Format a domain's known resources as a bullet list for prompt grounding."""
    return "\n".join(f"- {item}" for item in KNOWN_RESOURCES[domain])


def _domain_chain(domain: str, persona: str, schema: type[BaseModel]) -> Runnable:
    """Build a domain expert chain: a grounded system prompt piped into structured output."""
    system_message = (
        persona + " Only cite schemes or helplines from the list below -- do not invent "
        "names or numbers that are not in it.\n\nKnown " + domain + " resources:\n{resources}"
    )
    prompt = ChatPromptTemplate.from_messages(
        [("system", system_message), CASE_HUMAN_MESSAGE]
    ).partial(resources=_resource_block(domain))
    return prompt | llm.with_structured_output(schema)


health_chain = _domain_chain(
    "health",
    "You are a community health worker advising informal-sector workers in India.",
    HealthAssessment,
)
financial_chain = _domain_chain(
    "financial",
    "You are a financial-inclusion counselor advising informal-sector workers in India.",
    FinancialAssessment,
)
legal_chain = _domain_chain(
    "legal",
    "You are a paralegal advising informal-sector workers in India on their labor and consumer rights.",
    LegalAssessment,
)

DOMAIN_CHAINS: dict[str, Runnable] = {
    "health": health_chain,
    "financial": financial_chain,
    "legal": legal_chain,
}


## Step 3: Run only the relevant domain chains, concurrently

`RunnableParallel` is built from whichever chains triage marked relevant and invoked once -- LangChain runs its branches concurrently, so a case touching all three domains costs one round-trip's worth of wall-clock time, not three sequential ones.


In [ ]:
def run_domain_assessments(case: CrisisCase, triage: DomainTriage) -> dict[str, BaseModel]:
    """Run only the domain chains triage marked relevant, concurrently via RunnableParallel."""
    relevance = {
        "health": triage.health_relevant,
        "financial": triage.financial_relevant,
        "legal": triage.legal_relevant,
    }
    active_chains = {domain: chain for domain, chain in DOMAIN_CHAINS.items() if relevance[domain]}
    if not active_chains:
        return {}
    return RunnableParallel(active_chains).invoke(case.as_prompt_vars())


## Step 4: Cross-domain synthesis with reasoning

The domain assessments are correct in isolation but say nothing about how they compound each other. `synthesis_draft_chain` spends `reasoning_effort="high"` reasoning over all of them together, in plain text, so the model's full chain of thought is available afterward as `reasoning_content`. A second, lightweight call then fits that already-reasoned answer into `CrisisResolutionPlan` -- keeping the expensive reasoning call in the plain-text mode it is documented for, rather than asking it to reason and emit a tool call at the same time.


In [ ]:
class CrisisResolutionPlan(BaseModel):
    summary: str = Field(description="One or two sentences summarizing the overall situation")
    cross_domain_risks: list[str] = Field(
        description="Ways the domain issues compound each other, e.g. a medical leave causing a loan default"
    )
    immediate_actions: list[str] = Field(
        description="Prioritized actions for the next 24-48 hours, across all domains"
    )
    short_term_actions: list[str] = Field(description="Prioritized actions for the next few weeks")
    long_term_actions: list[str] = Field(
        description="Actions for lasting resilience, e.g. insurance enrolment or registrations"
    )
    escalation_contacts: list[str] = Field(
        description="Specific helplines or offices to contact, drawn only from the assessments or resources given"
    )


ALL_RESOURCES_TEXT = "\n\n".join(
    f"{domain.capitalize()}:\n{_resource_block(domain)}" for domain in KNOWN_RESOURCES
)

SYNTHESIS_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a caseworker who resolves crises for informal-sector workers in India by "
            "combining separate domain assessments into one coherent plan. Your job is "
            "specifically to find the interactions between domains -- for example, a "
            "hospitalization that stops income, which then risks a loan default -- and to "
            "prioritize actions accordingly. Only cite schemes or helplines that appear in the "
            "assessments or the resources below; do not invent new ones.\n\n"
            "All known resources:\n{all_resources}",
        ),
        (
            "human",
            "Case narrative:\n{narrative}\n\n"
            "Domain assessments (only relevant domains are present):\n{domain_assessments}",
        ),
    ]
).partial(all_resources=ALL_RESOURCES_TEXT)

synthesis_draft_chain = SYNTHESIS_PROMPT | reasoning_llm

STRUCTURE_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Convert the crisis resolution plan below into the requested structure. Do not "
            "add, remove, or reinterpret any information -- only reorganize it.",
        ),
        ("human", "{draft_plan}"),
    ]
)
structure_chain = STRUCTURE_PROMPT | llm.with_structured_output(CrisisResolutionPlan)


def synthesize_plan(
    case: CrisisCase, domain_results: dict[str, BaseModel]
) -> tuple[CrisisResolutionPlan, str | None]:
    """Reason over every domain assessment together, then structure the result."""
    domain_assessments_text = "\n\n".join(
        f"{domain.upper()} ASSESSMENT:\n{assessment.model_dump_json(indent=2)}"
        for domain, assessment in domain_results.items()
    ) or "No domain was triaged as relevant; treat this as a false alarm and say so in the summary."

    draft = synthesis_draft_chain.invoke(
        {"narrative": case.narrative, "domain_assessments": domain_assessments_text}
    )
    reasoning_trace = draft.additional_kwargs.get("reasoning_content")
    plan = structure_chain.invoke({"draft_plan": draft.content})
    return plan, reasoning_trace


## Step 5: Orchestrate the full pipeline

`resolve_crisis` is the one function a caller needs: triage, then the relevant domain chains, then synthesis.


In [ ]:
def resolve_crisis(case: CrisisCase) -> dict:
    """Run triage, the relevant domain chains, and cross-domain synthesis end to end."""
    triage = triage_chain.invoke(case.as_prompt_vars())
    domain_results = run_domain_assessments(case, triage)
    plan, reasoning_trace = synthesize_plan(case, domain_results)

    return {
        "case": case.model_dump(),
        "triage": triage.model_dump(),
        "domain_assessments": {domain: result.model_dump() for domain, result in domain_results.items()},
        "reasoning_trace": reasoning_trace,
        "plan": plan.model_dump(),
    }


## Run it: a crisis that spans all three domains

Ramesh's case touches health (the fracture), financial (the missed EMI), and legal (risk of repossession) all at once -- the scenario this notebook's architecture exists for.


In [ ]:
case_1 = CrisisCase(
    profile=WorkerProfile(
        occupation="auto-rickshaw driver",
        location="Bengaluru, Karnataka",
        monthly_income_range_inr="9000-12000",
        dependents=3,
    ),
    narrative=(
        "Ramesh fractured his leg in a road accident three days ago and is admitted at a "
        "government hospital. He has no health insurance. He still owes EMI on the loan he "
        "took to buy his auto-rickshaw, and the finance company has already called twice "
        "asking why this month's payment is late. He cannot drive for at least six weeks and "
        "is worried the company will repossess the vehicle. His wife and two school-going "
        "children depend on his income."
    ),
)

result_1 = resolve_crisis(case_1)

relevant_domains_1 = [
    d.removesuffix("_relevant")
    for d in ("health_relevant", "financial_relevant", "legal_relevant")
    if result_1["triage"][d]
]
print(f"Domains triaged as relevant: {relevant_domains_1}")
print()
print("Cross-domain risks identified:")
for risk in result_1["plan"]["cross_domain_risks"]:
    print(f"  - {risk}")
print()
print("Immediate actions:")
for action in result_1["plan"]["immediate_actions"]:
    print(f"  - {action}")

output_path_1 = OUTPUT_DIR / "case_1_auto_driver.json"
output_path_1.write_text(json.dumps(result_1, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nSaved full trace (including reasoning_trace) to {output_path_1}")


## Run it: a single-domain crisis

Sunita's case is purely financial. Triage should skip the health and legal chains entirely -- a direct, observable payoff of Step 1's dynamic routing.


In [ ]:
case_2 = CrisisCase(
    profile=WorkerProfile(
        occupation="street vendor",
        location="Pune, Maharashtra",
        monthly_income_range_inr="6000-8000",
        dependents=1,
    ),
    narrative=(
        "Sunita sells vegetables from a handcart. Unseasonal heavy rain last week destroyed "
        "almost all of her stock and damaged her cart. She has no savings left to buy fresh "
        "stock and has had to stop working for the past four days. She is in good health and "
        "has no ongoing loan or legal dispute."
    ),
)

result_2 = resolve_crisis(case_2)

relevant_domains_2 = [
    d.removesuffix("_relevant")
    for d in ("health_relevant", "financial_relevant", "legal_relevant")
    if result_2["triage"][d]
]
print(f"Domains triaged as relevant: {relevant_domains_2}")
print()
print("Immediate actions:")
for action in result_2["plan"]["immediate_actions"]:
    print(f"  - {action}")

output_path_2 = OUTPUT_DIR / "case_2_street_vendor.json"
output_path_2.write_text(json.dumps(result_2, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nSaved full trace to {output_path_2}")


## Next steps

- Try your own case narratives -- the triage step means you never have to hardcode which domains apply.
- Compare `reasoning_llm` with `reasoning_effort` set to `"low"`, `"medium"`, and `None`, and look at how `reasoning_trace` and the specificity of `cross_domain_risks` change.
- Extend `KNOWN_RESOURCES` with state-specific schemes (e.g. your state's BOCW welfare board contact details) for the regions you actually serve.
- Add a fourth domain (e.g. housing or workplace safety) by following the same pattern: a Pydantic schema, a grounded system prompt, and one new entry in `DOMAIN_CHAINS`.
- This notebook demonstrates orchestration patterns, not verified advice -- confirm every scheme name, eligibility rule, and helpline number against an official source before sharing it with a real worker.

Read more in [Build with Sarvam AI in LangChain](https://docs.sarvam.ai/api/integration/langchain), the [Chat Completions overview](https://docs.sarvam.ai/api/api-guides-tutorials/chat-completion/overview), and [Reasoning effort](https://docs.sarvam.ai/api/api-guides-tutorials/chat-completion/how-to/adjust-the-models-thinking-level).
